### 1.1 : Préparer les bibliothèques et les chemins

L'objectif est d'explorer le dataset image afin de récupérer les caractéristiques
principales de chaque fichier.

Pour chaque image, nous allons relever :

- son nom ;
- sa classe ;
- son format ;
- son mode ;
- sa largeur et sa hauteur ;
- l'écart-type de ses pixels ;
- son nombre de canaux ;
- sa taille en octets.

Les fichiers corrompus seront également pris en compte afin qu'ils ne bloquent
pas l'exploration du dataset.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError


RAW_DIR = Path("../data/raw")

CLASSES = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash",
]

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".gif",
    ".webp",
    ".tif",
    ".tiff",
}

In [ ]:
#verif 

print(f"Dossier du dataset : {RAW_DIR}")
print(f"Classes : {CLASSES}")

### 1.2 : Fonction d'analyse d'une image

La fonction `analyser_image()` inspecte une image sans la modifier.

Elle récupère ses métadonnées et calcule l'écart-type des pixels.

Si l'image est corrompue ou illisible, la fonction retourne tout de même
les informations disponibles et indique que l'image est corrompue.

In [ ]:
def analyser_image(image_path, classe):
    """Analyse une image et retourne ses caractéristiques."""
    
    taille_octets = image_path.stat().st_size

    resultat = {
        "nom": image_path.name,
        "classe": classe,
        "format": None,
        "mode": None,
        "largeur": None,
        "hauteur": None,
        "ecart_type_pixels": None,
        "nombre_canaux": None,
        "taille_octets": taille_octets,
        "corrompue": False,
    }

    try:
        # premiere ouverture pour verifier l'integrite du fichier
        with Image.open(image_path) as image:

            # verifier si l'image est exploitable
            image.verify()

        # deuxieme ouverture pour acceder reellement aux pixels
        with Image.open(image_path) as image:
            resultat["format"] = image.format
            resultat["mode"] = image.mode
            resultat["largeur"], resultat["hauteur"] = image.size

            pixels = np.asarray(image)

            if pixels.ndim == 2:
                resultat["nombre_canaux"] = 1
            elif pixels.ndim == 3:
                resultat["nombre_canaux"] = pixels.shape[2]

            resultat["ecart_type_pixels"] = float(pixels.std())

    except (UnidentifiedImageError, OSError, SyntaxError):
        resultat["corrompue"] = True

    return resultat